In [ ]:
import mcstasscript as ms

In [ ]:
import LET_functions

union_detector = True
instrument = LET_functions.make_instrument(union_detector=union_detector, air=False)

sample = instrument.add_component("sample", "Union_cylinder", after="sample_position")
sample.set_parameters(radius=0.01, yheight=0.03, priority=1000, material_string='"Bi"')
sample.set_AT(0, RELATIVE="sample_position")
sample.set_ROTATED([0, 60, 0], RELATIVE="sample_position")

cryostat_shell = instrument.add_component("cryostat_shell", "Union_cylinder", after="sample")
cryostat_shell.set_parameters(radius=0.08, yheight=0.3, material_string='"Al"', priority=990, p_interact=0.2)
cryostat_shell.set_AT(0,sample)

cryostat_vacuum = instrument.add_component("shell_vacuum", "Union_cylinder", after="cryostat_shell")
cryostat_vacuum.set_parameters(radius=cryostat_shell.radius-0.3E-3, yheight=cryostat_shell.yheight-0.01,
                               material_string='"Vacuum"', priority=991)
cryostat_vacuum.set_AT(0,cryostat_shell)
                               
# 30 by 50 cm at detector tubes
# sides 1 cm al box

beamstop = instrument.add_component("beamstop", "Union_box", after="sample_position")
beamstop.set_parameters(xwidth=0.3, yheight=0.5, zdepth=0.1, priority=1001, material_string='"B4C"', p_interact=0.1)
beamstop.set_AT(3.3, RELATIVE="sample_position")

beamstop_vacuum = instrument.add_component("beamstop_vacuum", "Union_box", after=beamstop)
beamstop_vacuum.set_parameters(xwidth=0.3-0.02, yheight=0.5-0.02, zdepth=0.1, priority=1002, material_string='"Vacuum"')
beamstop_vacuum.set_AT(-0.01, RELATIVE=beamstop)

instrument.show_parameters()

In [ ]:
#instrument.show_instrument()

In [ ]:
instrument.settings(ncount=1E8, mpi=10, suppress_output=False)

instrument.set_parameters(E0=12, dE=2.5)
#instrument.set_parameters(E0=7, dE=0.2)

data = instrument.backengine()

#data = ms.load_data("LET_29")
data

In [ ]:
scatter_logger = ms.name_search("logger_zx", data)
ms.make_plot(scatter_logger, log=True, orders_of_mag=20)

In [ ]:
import numpy as np
import copy

if union_detector:
    # Union detector workflow
    first_abs_logger = ms.name_search("module_0_abs_logger", data)
    
    tubes = []
    for mon in data[0:]:
        if "abs_logger" in mon.name:
            if hasattr(mon, "Events"):
                tubes.append(mon.Events)
        
    total = np.concatenate(tuple(tubes))

    p_array = total[:, first_abs_logger.find_variable_index("p")]
    t_array = total[:, first_abs_logger.find_variable_index("t")]
    
    x_array = total[:, first_abs_logger.find_variable_index("x")]
    y_array = total[:, first_abs_logger.find_variable_index("y")]
    z_array = total[:, first_abs_logger.find_variable_index("z")]
    position = total[:, first_abs_logger.find_variable_index("x"):first_abs_logger.find_variable_index("z")+1]

else: 
    # Monitor_nD workflow
    detector = ms.name_search("detector", data)
    print(detector)

    p_array = detector.get_data_column("p")
    t_array = detector.get_data_column("t")
    x_array = detector.get_data_column("x")
    y_array = detector.get_data_column("y")
    z_array = detector.get_data_column("z")

    position = np.stack((x_array.T, y_array.T, z_array.T)).T
    print(position.shape)

In [ ]:
import scipp as sc

da = sc.DataArray(
data=sc.array(dims=["events"], values=p_array, unit=sc.units.counts),
coords={
  "t": sc.array(dims=["events"], values=t_array, unit="s"),
  "x": sc.array(dims=["events"], values=x_array, unit="m"),    
  "y": sc.array(dims=["events"], values=y_array, unit="m"),
  "z": sc.array(dims=["events"], values=z_array, unit="m"),    
  'source_position': sc.vector([0, 0, -instrument.get_component("source").dist], unit='m'),
  'sample_position': sc.vector([0,0,0], unit='m'),
  'position': sc.vectors(dims=['events'], values=position, unit='m'),
  },
)

# Add two theta
th = np.atan2(da.coords["x"].values, da.coords["z"].values)*180/3.14159
da.coords["th"] = sc.array(dims=["events"], values=th, unit="deg")

In [ ]:
# Check the limits
for variable in ["x", "y", "z", "t", "th"]:
    print(variable.ljust(20), str(da.coords[variable].min().value).ljust(30), str(da.coords[variable].max().value).ljust(20))

In [ ]:
import plopp as pp
%matplotlib widget

times = sc.linspace("t", start=0.002, stop=0.015, num=80, unit="s")
print(times)

pp.slicer(da.hist(y=50, th=200, t=times), keep=["th", "y"])

In [ ]:
import plopp as pp
%matplotlib widget

times = sc.linspace("t", start=0.002, stop=0.03, num=80, unit="s")
print(times)

da_hist = da.hist(y=50, th=200, t=times)
da_hist.data = sc.log10(da_hist.data//sc.scalar(value=1E-5, unit="counts"))

pp.slicer(da_hist, keep=["th", "y"], vmax=6, vmin=4, cmap="plasma")

In [ ]:
da

In [ ]:
import plopp as pp

pp.scatter3dfigure(pp.Node(da), size=0.00002, cbar=True, norm="log", vmin=1E-4)

In [ ]:
from scippneutron.conversion.graph.beamline import beamline
from scippneutron.conversion.graph.tof import elastic

# McStas provides absolute time, not time of flight
da.coords["tof"] = da.coords["t"]

graph = {**beamline(scatter=True), **elastic("tof")}

In [ ]:
da = da.transform_coords("energy", graph=graph)
da = da.transform_coords("wavelength", graph=graph)
da = da.transform_coords("dspacing", graph=graph)

In [ ]:
d_bins = sc.linspace("dspacing", start=0.1, stop=2.5, num=600, unit="Å")
print(d_bins)
da.hist(dspacing=d_bins).plot(norm="linear", figsize=(10,4))

In [ ]:
da

In [ ]:
import plopp as pp
%matplotlib widget

times = sc.linspace("t", start=0.002, stop=0.005, num=80, unit="s")
energies = sc.linspace("energy", start=6, stop=8, num=1000, unit="meV")
print(times)

da_hist = da.hist(y=50, th=200, energy=energies)
da_hist.data = sc.log10(da_hist.data//sc.scalar(value=1E-5, unit="counts"))

pp.slicer(da_hist, keep=["th", "y"])

In [ ]:
%matplotlib inline

energies = sc.linspace("energy", start=6, stop=8, num=1000, unit="meV")
da.hist(energy=energies).plot(figsize=(10,6))